# 五个新问题检查

我保持混合检索算法和参数不变，用五个没有用于此前调试的问题检查，并保存了本次结果与五条人工评价。下面保留原本地练习的实际输出，重新运行会生成新文件。

## 题目与核对标准

检查负载条件、错误前提、按钮方向和资料缺失。标准只供事后核对，不发送给模型。先安装仓库根目录依赖，打开本Notebook并选择对应Python内核。

In [1]:
import json
import hashlib
try:
    import manual_project as mp
    import retrieval_upgrade as up
except ModuleNotFoundError:
    from public_manual import manual_project as mp
    from public_manual import retrieval_upgrade as up

records = mp.prepare_documents()
model = mp.load_search_model()
doc_vectors = mp.encode_documents(records, model)
tools = up.build_search_tools(records, model, doc_vectors)
cases09 = json.loads((mp.BASE_DIR / "unseen_cases09.json").read_text(encoding="utf-8"))

# 保存检索程序的版本指纹，说明这一轮使用的是哪个版本。
code_sha256 = hashlib.sha256((mp.BASE_DIR / "retrieval_upgrade.py").read_bytes()).hexdigest()
for case in cases09:
    print(case["id"], case["question"])


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

U01 手册列出的额定负载与低于额定负载时的工作温度范围分别是多少？
U02 100%负载和40%负载时，输入电压范围分别是多少？
U03 电池模块放在仓库里就不用再充电了，对吗？手册对此怎么规定？
U04 查看事件日志时，想看更早的事件和更新的事件，分别用哪个方向按钮？
U05 这台UPS今天在苏州经销商处的售价是多少元？


## 运行与记录

下一格请求DeepSeek五次，每题结果逐步保存。新运行使用带时间的文件名，保留既有记录；结果不理想也保留，不反复请求后只挑正确答案。

In [2]:
from getpass import getpass
from datetime import datetime
from openai import OpenAI

api_key = getpass("DeepSeek API key（隐藏输入）：").strip()
if not api_key:
    raise ValueError("请输入密钥")
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com", timeout=60, max_retries=0)
run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
output_name = f"llm09_new5_{run_id}.json"
new_results = up.run_cases(cases09, tools, client, output_name)
for result in new_results:
    result["retrieval_code_sha256"] = code_sha256
mp.save_json(new_results, output_name)
print("本次文件：", mp.BASE_DIR / "results" / output_name)


U01 ok 额定负载时工作温度为 0 到 40°C；低于额定负载时为 40 到 50°C。依据原文：“工作 额定负载时 0 到 40°C / 低于额定负载时 40 到 50 °C”。来源：[EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第4页，块01]。 

U02 ok 100%负载时输入电压范围为 160 V AC – 280 V AC；40%负载时为 110 V AC – 285 V AC。  
依据：  
- “输入电压范围(100%负载) 160 V AC – 280 V AC”  
- “输入电压范围(40%负载) 110 V AC – 285V AC”  
来源：[EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第5页，块01] 

U03 ok 不对。手册未支持“电池模块放在仓库里就不用再充电”的说法。

手册摘录只提供了电池类型、模块数量、电压和 Ah 额定值等电池规格，以及更换电池需参阅适用电池更换用户手册/联系经销商或 APC 网站的信息；并未规定仓库储存时无需再充电。

资料未提供关于电池模块仓库储存是否需要充电的明确规定。 

U04 ok 查看更早的事件按**向下按钮**，查看更新的事件按**向上按钮**。原文：“向下按钮导航至旧事件，向上按钮导航至新事件。”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第16页，块02） 

U05 ok 资料未提供。手册摘录只涉及 UPS 设置、电池电压/容量和前面板说明，没有苏州经销商售价信息。 

本次文件： D:\1rance\LLM\LLM_07_公开手册问答项目\results\llm09_new5_20261013_121225_300020.json


## 核对原文、回答和引用

温度与电压要区分负载条件；贮存条款在第4页，要求每六个月充电一次；按钮方向在第16页；静态手册没有当天当地售价。下面只查看一条，修改index可以查看其他结果，不请求模型。

In [3]:
index = 0
result = new_results[index]
print(result["id"], result["question"])
print("核对标准：", result["expected"])
print("实际回答：", result["answer"])
up.show_hits(result["retrieved"])


U01 手册列出的额定负载与低于额定负载时的工作温度范围分别是多少？
核对标准： 额定负载时0到40°C；低于额定负载时40到50°C；应保留负载条件。
实际回答： 额定负载时工作温度为 0 到 40°C；低于额定负载时为 40 到 50°C。依据原文：“工作 额定负载时 0 到 40°C / 低于额定负载时 40 到 50 °C”。来源：[EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第4页，块01]。
EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第4页，块01
规格
环境规格
注意
设备损坏风险
• UPS 必须在室内使用。
• 安装位置应坚固，足以支撑 UPS 的重量。
• 不要在灰尘过多或温度或湿度超出规定的地方使用 UPS。
否则，可能导致设备损坏。
温度 工作 额定负载时 0 到 40°C
低于额定负载时 40 到 50 °C
本设备仅适用于在室内使用。应
将其放置在牢固可靠并足以支撑
其重量的位置。
不要在灰尘过多或温度或湿度超
出规定的地方使用 UPS。
注：贮存期间每六个月给电池模
块充电一次。
贮存 -20 到 60°C
海拔高度 工作 0 - 2,000

EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第5页，块02
UPS 容量 1000 V A / 800 W 2000 V A / 1600 W 3000 V A / 2400 W
额定输出电压 230 V AC
其它可编程电压 220 V AC, 240 V AC
额定负载时效率 最大 88%
输出电压调节 ± 1% 静态
输出电压失真
• 完全线性负载时最大 3%，
• 完全非线性 RCD 负载（100% V A，0.9 PF）时最大 6%
• 最后 60 秒备份时间时 15%(完全负载仅适用于内部电池)
频率 － 电池模式 50 Hz ± 0.5% 或 60 Hz ±

EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第4页，块02
用 UPS。
注：贮存期间每六个月给电池模
块充电一次。
贮存 -20 到 60°C
海拔高度 工作 0 - 2,000 m: 正常操作
> 2,000 m：高度每增加 100 m 负载减小
1%
> 3,000 m：UPS 不工作
贮存

### 本次五条人工评价

以下是本次实际判断，四条通过，U03不通过。重新运行后应对照新回答、新候选重新填写，再保存评价。

In [4]:
# 在这里给5条结果添加review
new_results[0]["review"] = "通过：两种负载条件与对应温度范围均正确，第4页块01包含完整依据。"

new_results[1]["review"] = "通过：100%与40%负载下的输入电压范围均正确，第5页块01包含对应依据。"

new_results[2]["review"] = "不通过：未检索到第4页的贮存充电条款，没有回答每六个月充电一次；问题出在漏检。"

new_results[3]["review"] = "通过：向下查看旧事件、向上查看新事件，回答与第16页块02一致。"

new_results[4]["review"] = "通过：资料没有提供当天苏州经销商的售价，模型没有编造金额。"

## 保存人工评价

先填写评价，再运行保存。写入字典不会自动更新JSON；这格不请求模型。

In [5]:
print("已填写评价数量：", sum(bool(result.get("review", "").strip()) for result in new_results))
result_path = mp.save_json(new_results, output_name)
print("结果已保存：", result_path)


已填写评价数量： 5
结果已保存： D:\1rance\LLM\LLM_07_公开手册问答项目\results\llm09_new5_20261013_121225_300020.json


## 本次观察

四条通过，U03漏掉了第4页的贮存充电依据，未回答每六个月充电一次。接口完成、命中相关页、回答满足标准是三件事。

In [ ]:
# 本次人工核对4/5通过，U03不通过，问题在漏检贮存充电规定。
# 前一轮修复的三个案例答对后，新问题仍漏检，当前方案还需要新问题检查。

## 记录与版本

本次保存结果为 `results/llm09_new5_20261013_121225_300020.json`，题目为 `unseen_cases09.json`。保留失败和人工评价，不新增检索规则。

结果中的程序指纹对应原本地运行版本。仓库模块做了导入路径和旧基线文件名适配，文件指纹不同，核心检索逻辑相同；重新运行时会记录仓库当前文件的指纹。

这五个问题一旦用于调试，就不再作为下一轮未见过的新问题。4/5是本次人工核对结果，没有自动评分，也不是全面评测。